# Exchange Rate Regime Stratification, Diversification, and Confidence Intervals

This notebook extends the terminal liability analysis from `consolidated_currency_analysis.ipynb` with three additions:

1. **Regime stratification** — splitting the sample into EUR-linked (CFA franc zone + Comoros, Cape Verde), ZAR-linked (Common Monetary Area), and flexible-rate countries
2. **Country-level vs. pooled premiums** — quantifying the heterogeneity in risk and the diversification benefit of a pan-African facility
3. **Bootstrap confidence intervals** — block-bootstrapping countries to get standard errors on premium estimates

In [ ]:
%run ../source/data_analysis_functions.py
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

hybrid = build_hybrid_index_and_ma_df(start="2000-01-01", end="2023-12-31")
hybrid['regime'] = hybrid['country'].map(fx_regime).fillna('Flexible')

print(f"Total country-year observations: {len(hybrid)}")
print(f"Countries: {hybrid['country'].nunique()}")
print(f"\nCountries by regime:")
for regime in ['EUR-linked', 'ZAR-linked', 'Flexible']:
    countries = sorted(hybrid[hybrid['regime'] == regime]['country'].unique())
    print(f"  {regime} ({len(countries)}): {', '.join(countries)}")

## 1. Terminal residuals by exchange rate regime

The EUR-linked group (16 countries) experienced a cluster of real depreciation events around 2014-2017, driven by the EUR/USD decline rather than domestic factors. This means:
- Their events are **highly correlated** with each other
- The nature of the risk is different (imported anchor-currency movements vs. domestic devaluation)
- A facility covering only EUR-linked countries would face concentrated, correlated risk

The flexible-rate group (33 countries) includes the largest individual events (Nigeria, Angola, Sudan, Ghana) but with more idiosyncratic timing.

In [ ]:
residuals = compute_terminal_residuals(hybrid, years=15, discount_rate=0.1, window_mode="rolling")
residuals['regime'] = residuals['country'].map(fx_regime).fillna('Flexible')

# Overall
print("=" * 70)
print("OVERALL (ALL COUNTRIES)")
print("=" * 70)
print(f"  Annual premium (E[loss]):  {residuals['loss'].mean()*100:.3f}% of debt stock")
print(f"  Probability of loss:       {(residuals['loss'] < 0).mean()*100:.1f}%")
print(f"  Mean loss given loss:      {residuals.loc[residuals['loss']<0, 'loss'].mean()*100:.3f}%")
print(f"  Worst window:              {residuals['loss'].min()*100:.3f}%")
print(f"  N country-windows:         {len(residuals)}")
print(f"  N countries:               {residuals['country'].nunique()}")

# By regime
print("\n" + "=" * 70)
print("BY EXCHANGE RATE REGIME")
print("=" * 70)

regime_stats = []
for regime in ['EUR-linked', 'ZAR-linked', 'Flexible']:
    r = residuals[residuals['regime'] == regime]
    stats = {
        'Regime': regime,
        'Countries': r['country'].nunique(),
        'Windows': len(r),
        'Premium (%)': r['loss'].mean() * 100,
        'P(loss) (%)': (r['loss'] < 0).mean() * 100,
        'E[loss|loss] (%)': r.loc[r['loss'] < 0, 'loss'].mean() * 100 if (r['loss'] < 0).any() else 0,
        'Worst (%)': r['loss'].min() * 100,
    }
    regime_stats.append(stats)

regime_df = pd.DataFrame(regime_stats)
print(regime_df.to_string(index=False, float_format='%.3f'))

## 2. Country-level premiums vs. pooled premium

The pooled premium averages across all country-windows. Individual country premiums vary widely — some countries have no losses in any 15-year window, while others face substantial expected losses.

The diversification benefit of pooling is not in the expected loss (which is additive) but in the **variance reduction**: a facility covering many countries faces a tighter loss distribution than one covering a single country, reducing capital requirements.

In [ ]:
# Country-level premiums
country_stats = residuals.groupby('country').agg(
    n_windows=('loss', 'count'),
    premium=('loss', 'mean'),
    prob_loss=('loss', lambda x: (x < 0).mean()),
    worst=('loss', 'min'),
    best=('net_pv_by_debt_stock', 'max'),
).reset_index()
country_stats['regime'] = country_stats['country'].map(fx_regime).fillna('Flexible')
country_stats = country_stats.sort_values('premium')

pooled = residuals['loss'].mean()

print(f"Pooled annual premium: {pooled*100:.3f}%\n")
print(f"Country-level premium distribution:")
print(f"  Mean:   {country_stats['premium'].mean()*100:.3f}%")
print(f"  Median: {country_stats['premium'].median()*100:.3f}%")
print(f"  Std:    {country_stats['premium'].std()*100:.3f}%")
print(f"  Min:    {country_stats['premium'].min()*100:.3f}% ({country_stats.iloc[0]['country']})")
print(f"  Max:    {country_stats['premium'].max()*100:.3f}% ({country_stats.iloc[-1]['country']})")
print(f"  Countries with zero loss: {(country_stats['premium'] == 0).sum()}")

# Show full table
display_df = country_stats[['country', 'regime', 'n_windows', 'premium', 'prob_loss', 'worst']].copy()
display_df.columns = ['Country', 'Regime', 'Windows', 'Premium (%)', 'P(loss)', 'Worst (%)']
display_df['Premium (%)'] = display_df['Premium (%)'] * 100
display_df['Worst (%)'] = display_df['Worst (%)'] * 100
display_df['P(loss)'] = display_df['P(loss)'] * 100
print("\n" + display_df.to_string(index=False, float_format='%.3f'))

In [ ]:
# Visualization: country premiums by regime
regime_colors = {'EUR-linked': '#4c78a8', 'ZAR-linked': '#f58518', 'Flexible': '#54a24b'}

fig, ax = plt.subplots(figsize=(14, 5))
cs = country_stats.reset_index(drop=True)
colors = [regime_colors[r] for r in cs['regime']]
ax.bar(range(len(cs)), cs['premium'] * 100, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(pooled * 100, color='red', linestyle='--', linewidth=1, label=f'Pooled premium ({pooled*100:.3f}%)')
ax.set_xticks(range(len(cs)))
ax.set_xticklabels(cs['country'], rotation=90, fontsize=7)
ax.set_ylabel('Annual premium (% of debt stock)')
ax.set_title('Country-Level FX Risk Premium by Exchange Rate Regime')

# Legend
from matplotlib.patches import Patch
handles = [Patch(facecolor=regime_colors[r], label=r) for r in ['EUR-linked', 'ZAR-linked', 'Flexible']]
handles.append(plt.Line2D([0], [0], color='red', linestyle='--', label=f'Pooled ({pooled*100:.3f}%)'))
ax.legend(handles=handles, loc='lower left', fontsize=8)

plt.tight_layout()
plt.savefig('../output/country_premiums_by_regime.png', dpi=150, bbox_inches='tight')
plt.show()

### Correlation structure

The diversification benefit depends on the correlation of real exchange rate movements across countries. We expect:
- **High within-group correlation** for EUR-linked countries (driven by the common EUR/USD rate)
- **Lower correlation** among flexible-rate countries (idiosyncratic domestic factors)
- **Low cross-group correlation** between EUR-linked and flexible-rate countries

In [ ]:
# Correlation of annual H changes across countries
wide = hybrid.pivot(index='year', columns='country', values='fx_index')
returns = wide.pct_change().dropna()
corr = returns.corr()

eur_countries = sorted([c for c in corr.columns if fx_regime.get(c) == 'EUR-linked'])
zar_countries = sorted([c for c in corr.columns if fx_regime.get(c) == 'ZAR-linked'])
flex_countries = sorted([c for c in corr.columns if c not in eur_countries + zar_countries])

def mean_pairwise_corr(corr_matrix, group):
    sub = corr_matrix.loc[group, group]
    mask = np.triu(np.ones(sub.shape, dtype=bool), k=1)
    vals = sub.where(mask).stack()
    return vals.mean(), vals.std(), len(vals)

def mean_cross_corr(corr_matrix, g1, g2):
    vals = corr_matrix.loc[g1, g2].stack()
    return vals.mean(), vals.std(), len(vals)

print("Mean pairwise correlation of annual H changes:")
print("=" * 55)

m, s, n = mean_pairwise_corr(corr, eur_countries)
print(f"Within EUR-linked:           {m:.3f} (sd {s:.3f}, n={n} pairs)")

if len(zar_countries) > 1:
    m, s, n = mean_pairwise_corr(corr, zar_countries)
    print(f"Within ZAR-linked:           {m:.3f} (sd {s:.3f}, n={n} pairs)")

m, s, n = mean_pairwise_corr(corr, flex_countries)
print(f"Within Flexible:             {m:.3f} (sd {s:.3f}, n={n} pairs)")

m, s, n = mean_cross_corr(corr, eur_countries, flex_countries)
print(f"EUR-linked vs Flexible:      {m:.3f} (sd {s:.3f}, n={n} pairs)")

if len(zar_countries) > 0:
    m, s, n = mean_cross_corr(corr, zar_countries, flex_countries)
    print(f"ZAR-linked vs Flexible:      {m:.3f} (sd {s:.3f}, n={n} pairs)")
    m, s, n = mean_cross_corr(corr, eur_countries, zar_countries)
    print(f"EUR-linked vs ZAR-linked:    {m:.3f} (sd {s:.3f}, n={n} pairs)")

In [ ]:
# Correlation heatmap ordered by regime
ordered = eur_countries + zar_countries + flex_countries
corr_ordered = corr.loc[ordered, ordered]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_ordered.values, cmap='RdBu_r', vmin=-0.5, vmax=1, aspect='auto')
ax.set_xticks(range(len(ordered)))
ax.set_yticks(range(len(ordered)))
ax.set_xticklabels(ordered, rotation=90, fontsize=6)
ax.set_yticklabels(ordered, fontsize=6)

# Draw regime boundaries
b1 = len(eur_countries) - 0.5
b2 = len(eur_countries) + len(zar_countries) - 0.5
for b in [b1, b2]:
    ax.axhline(b, color='black', linewidth=1.5)
    ax.axvline(b, color='black', linewidth=1.5)

# Labels
mid_eur = len(eur_countries) / 2
mid_zar = len(eur_countries) + len(zar_countries) / 2
mid_flex = len(eur_countries) + len(zar_countries) + len(flex_countries) / 2
ax.text(-3, mid_eur, 'EUR', fontsize=9, fontweight='bold', ha='center', va='center', rotation=90)
ax.text(-3, mid_zar, 'ZAR', fontsize=9, fontweight='bold', ha='center', va='center', rotation=90)
ax.text(-3, mid_flex, 'Flex', fontsize=9, fontweight='bold', ha='center', va='center', rotation=90)

plt.colorbar(im, ax=ax, label='Correlation', shrink=0.8)
ax.set_title('Pairwise Correlation of Annual Real Exchange Rate Changes')
plt.tight_layout()
plt.savefig('../output/correlation_heatmap_by_regime.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Bootstrap confidence intervals

We use a **block bootstrap** resampling countries with replacement to preserve within-country serial correlation across windows. Each bootstrap replicate draws $N$ countries (with replacement) from the $N$ in the sample and computes the pooled premium from all their windows.

This gives us standard errors and 95% confidence intervals for:
- The overall pooled premium
- Regime-specific premiums
- The difference between regime premiums

In [ ]:
def bootstrap_premium(residuals, n_boot=10000, seed=42):
    """
    Block bootstrap (resampling countries) for the pooled premium.
    Returns dict with point estimate, SE, 95% CI, and full distribution.
    """
    rng = np.random.default_rng(seed)
    countries = residuals['country'].unique()
    n = len(countries)
    
    # Pre-index for speed
    country_losses = {c: residuals.loc[residuals['country'] == c, 'loss'].values for c in countries}
    
    boot = np.empty(n_boot)
    for b in range(n_boot):
        sampled = rng.choice(countries, size=n, replace=True)
        all_losses = np.concatenate([country_losses[c] for c in sampled])
        boot[b] = all_losses.mean()
    
    point = residuals['loss'].mean()
    return {
        'point': point,
        'se': boot.std(),
        'ci_lo': np.percentile(boot, 2.5),
        'ci_hi': np.percentile(boot, 97.5),
        'distribution': boot
    }

# Overall
print("=" * 70)
print("BOOTSTRAP CONFIDENCE INTERVALS (10,000 country-block resamples)")
print("=" * 70)

overall = bootstrap_premium(residuals)
print(f"\nOverall pooled premium:")
print(f"  Point estimate:  {overall['point']*100:.3f}%")
print(f"  Std error:       {overall['se']*100:.3f}%")
print(f"  95% CI:          [{overall['ci_lo']*100:.3f}%, {overall['ci_hi']*100:.3f}%]")

regime_boot = {}
for regime in ['EUR-linked', 'ZAR-linked', 'Flexible']:
    r = residuals[residuals['regime'] == regime]
    b = bootstrap_premium(r)
    regime_boot[regime] = b
    print(f"\n{regime} ({r['country'].nunique()} countries):")
    print(f"  Point estimate:  {b['point']*100:.3f}%")
    print(f"  Std error:       {b['se']*100:.3f}%")
    print(f"  95% CI:          [{b['ci_lo']*100:.3f}%, {b['ci_hi']*100:.3f}%]")

In [ ]:
# Bootstrap distribution plots
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)

configs = [
    ('All Countries (Pooled)', overall, 'steelblue'),
    ('EUR-linked', regime_boot['EUR-linked'], '#4c78a8'),
    ('ZAR-linked', regime_boot['ZAR-linked'], '#f58518'),
    ('Flexible', regime_boot['Flexible'], '#54a24b'),
]

for ax, (title, b, color) in zip(axes, configs):
    dist = b['distribution'] * 100
    ax.hist(dist, bins=60, color=color, edgecolor='white', linewidth=0.3, alpha=0.8)
    ax.axvline(b['point'] * 100, color='red', linestyle='--', linewidth=1.2,
               label=f"Point: {b['point']*100:.3f}%")
    ax.axvline(b['ci_lo'] * 100, color='gray', linestyle=':', linewidth=1)
    ax.axvline(b['ci_hi'] * 100, color='gray', linestyle=':', linewidth=1,
               label=f"95% CI: [{b['ci_lo']*100:.3f}, {b['ci_hi']*100:.3f}]")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Annual premium (% of debt)')
    ax.legend(fontsize=7)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

axes[0].set_ylabel('Frequency')
plt.suptitle('Bootstrap Distributions of Annual FX Risk Premium', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../output/bootstrap_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Sensitivity to discount rate

Check whether the regime-level findings are robust across discount rates.

In [ ]:
# Sensitivity: premiums across discount rates and regimes
rows = []
for dr in [0.01, 0.05, 0.10, 0.15]:
    res = compute_terminal_residuals(hybrid, years=15, discount_rate=dr, window_mode="rolling")
    res['regime'] = res['country'].map(fx_regime).fillna('Flexible')
    
    rows.append({
        'Discount rate': f"{dr:.0%}",
        'All': res['loss'].mean() * 100,
        'EUR-linked': res.loc[res['regime'] == 'EUR-linked', 'loss'].mean() * 100,
        'ZAR-linked': res.loc[res['regime'] == 'ZAR-linked', 'loss'].mean() * 100,
        'Flexible': res.loc[res['regime'] == 'Flexible', 'loss'].mean() * 100,
    })

sens_df = pd.DataFrame(rows)
print("Annual premium (% of debt stock) by discount rate and regime")
print("=" * 65)
print(sens_df.to_string(index=False, float_format='%.3f'))

## Summary

Key findings from this analysis:

In [ ]:
print("KEY FINDINGS")
print("=" * 70)
print(f"""                                    
1. REGIME HETEROGENEITY
   The annual FX risk premium differs substantially by exchange rate regime:
   - EUR-linked (CFA + Comoros, Cape Verde): {regime_boot['EUR-linked']['point']*100:.3f}%
     95% CI: [{regime_boot['EUR-linked']['ci_lo']*100:.3f}%, {regime_boot['EUR-linked']['ci_hi']*100:.3f}%]
   - ZAR-linked (Lesotho, Namibia, Eswatini): {regime_boot['ZAR-linked']['point']*100:.3f}%
     95% CI: [{regime_boot['ZAR-linked']['ci_lo']*100:.3f}%, {regime_boot['ZAR-linked']['ci_hi']*100:.3f}%]
   - Flexible (33 countries): {regime_boot['Flexible']['point']*100:.3f}%
     95% CI: [{regime_boot['Flexible']['ci_lo']*100:.3f}%, {regime_boot['Flexible']['ci_hi']*100:.3f}%]

2. COUNTRY-LEVEL HETEROGENEITY
   Individual country premiums range from {country_stats['premium'].min()*100:.3f}% to {country_stats['premium'].max()*100:.3f}%.
   {(country_stats['premium'] == 0).sum()} countries have zero expected loss over all 15-year windows.

3. CORRELATION AND DIVERSIFICATION
   EUR-linked countries are highly correlated (common EUR/USD driver).
   Flexible-rate countries are less correlated (idiosyncratic drivers).
   Cross-regime correlation is low, supporting diversification benefits
   from a pan-African pool.

4. OVERALL PREMIUM
   Pooled annual premium: {overall['point']*100:.3f}% of debt stock
   95% CI: [{overall['ci_lo']*100:.3f}%, {overall['ci_hi']*100:.3f}%]
""")